# DCAMF-Net 推理 Notebook

加载已训练权重，对测试集进行推理评估、可视化和结果汇总。不需要重新训练。

## 1. 环境与路径

In [ ]:
import os, sys, json, re, random
import torch, numpy as np, soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

os.environ['OMP_NUM_THREADS'] = '8'

PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'dcamf_net'))

print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Root: {PROJECT_ROOT}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 2. 加载 DCAMF-Net 权重

In [ ]:
from dcamf_net.model import DCAMFNet
from argparse import Namespace

args = Namespace(
    enc_channels=256, enc_kernel_size=80, enc_stride=40,
    chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
    ffn_hidden=512, dw_kernel_size=31, checkpoint=None
)

model = DCAMFNet(
    in_channels=1, enc_channels=256, enc_kernel_size=80, enc_stride=40,
    chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
    ffn_hidden=512, dw_kernel_size=31,
).to(device)

ckpt_path = PROJECT_ROOT / 'experiments' / 'dcamf_net' / 'checkpoints' / 'best_SISNR.pth'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state_dict = {k: v for k, v in ckpt.items()
              if not k.endswith('.total_ops') and not k.endswith('.total_params')
              and k not in ('total_ops', 'total_params')}
model.load_state_dict(state_dict)
model.eval()

total = sum(p.numel() for p in model.parameters())
print(f"DCAMF-Net loaded. {total:,} parameters.")
fw = ckpt.get('mask_fusion_weights')
if fw is not None:
    print(f"Fusion weights: {fw}")


## 3. 评估函数

In [ ]:
def compute_sisnr(est, ref):
    est = est.reshape(-1); ref = ref.reshape(-1)
    est = est - est.mean(); ref = ref - ref.mean()
    dot = (est * ref).sum()
    scale = dot / (ref * ref).sum() if (ref * ref).sum() > 1e-8 else 0
    target = scale * ref
    noise = est - target
    return 10 * np.log10((target**2).sum() / ((noise**2).sum() + 1e-8) + 1e-8)

def compute_sdr(est, ref):
    est = est.reshape(-1); ref = ref.reshape(-1)
    noise = ref - est
    return 10 * np.log10((ref**2).sum() / ((noise**2).sum() + 1e-8) + 1e-8)

def eval_model_on_testset(model, test_dir, device, desc="Eval"):
    clean_dir = Path(test_dir) / "clean"
    noisy_dir = Path(test_dir) / "noisy"
    files = sorted([f.name for f in clean_dir.glob("*.wav")])
    sisnr_list, sdr_list = [], []
    model.eval()
    with torch.no_grad():
        for fname in tqdm(files, desc=desc):
            clean, sr = sf.read(str(clean_dir / fname))
            noisy, _ = sf.read(str(noisy_dir / fname))
            if clean.ndim > 1: clean = clean.mean(axis=1)
            if noisy.ndim > 1: noisy = noisy.mean(axis=1)
            L = min(len(clean), len(noisy))
            clean, noisy = clean[:L], noisy[:L]
            noisy_t = torch.from_numpy(noisy).float().unsqueeze(0).unsqueeze(0).to(device)
            s_est = model(noisy_t).squeeze().cpu().numpy()  # model returns enhanced signal
            if len(s_est) > L: s_est = s_est[:L]
            elif len(s_est) < L: s_est = np.pad(s_est, (0, L - len(s_est)))
            # s_est is already the enhanced signal (model output = x - n_hat)
            sisnr_list.append(compute_sisnr(s_est, clean) - compute_sisnr(noisy, clean))
            sdr_list.append(compute_sdr(s_est, clean) - compute_sdr(noisy, clean))
    return {"si_snri": np.mean(sisnr_list), "sdri": np.mean(sdr_list), "n": len(files)}

# CRN-specific evaluation (STFT domain)
def eval_crn_stft(crn_net, test_dir, device, desc="CRN"):
    clean_dir = Path(test_dir) / "clean"
    noisy_dir = Path(test_dir) / "noisy"
    files = sorted([f.name for f in clean_dir.glob("*.wav")])
    sisnr_list, sdr_list = [], []
    win_size, hop_size = 512, 256
    window = torch.hann_window(win_size).to(device)
    with torch.no_grad():
        for fname in tqdm(files, desc=desc):
            clean, sr = sf.read(str(clean_dir / fname))
            noisy, _ = sf.read(str(noisy_dir / fname))
            if clean.ndim > 1: clean = clean.mean(axis=1)
            if noisy.ndim > 1: noisy = noisy.mean(axis=1)
            L = min(len(clean), len(noisy))
            clean, noisy = clean[:L], noisy[:L]
            noisy_t = torch.from_numpy(noisy).float().unsqueeze(0).to(device)
            stft = torch.stft(noisy_t, n_fft=win_size, hop_length=hop_size,
                               win_length=win_size, window=window, return_complex=True)
            mag = stft.abs().unsqueeze(1)
            phase = stft.angle()
            est_mag = crn_net(mag).squeeze(1)
            est_stft = est_mag * torch.exp(1j * phase)
            s_est_wav = torch.istft(est_stft, n_fft=win_size, hop_length=hop_size,
                                     win_length=win_size, window=window, length=L)
            s_est = s_est_wav.squeeze().cpu().numpy()
            if len(s_est) > L: s_est = s_est[:L]
            elif len(s_est) < L: s_est = np.pad(s_est, (0, L - len(s_est)))
            sisnr_list.append(compute_sisnr(s_est, clean) - compute_sisnr(noisy, clean))
            sdr_list.append(compute_sdr(s_est, clean) - compute_sdr(noisy, clean))
    return {"si_snri": np.mean(sisnr_list), "sdri": np.mean(sdr_list), "n": len(files)}

print("Evaluation functions ready.")


## 4. DCAMF-Net — ShipsEar 推理

In [ ]:
print('Evaluating DCAMF-Net on ShipsEar...')
dcamf_shipsear = {}
for tn in ['test1', 'test2', 'test3']:
    td = PROJECT_ROOT / 'data' / 'ShipsEar' / tn
    if td.exists():
        r = eval_model_on_testset(model, str(td), device, desc=f'  {tn}')
        dcamf_shipsear[tn] = r
        print(f"  {tn}: SI-SNRi={r['si_snri']:.2f} dB, SDRi={r['sdri']:.2f} dB")
    else:
        print(f"  {tn}: NOT FOUND")


## 5. DCAMF-Net — DeepShip 推理

In [ ]:
ds_dir = PROJECT_ROOT / 'data' / 'DeepShip' / 'test'
if ds_dir.exists():
    dcamf_deepship = eval_model_on_testset(model, str(ds_dir), device, desc='  DeepShip')
    print(f"DeepShip: SI-SNRi={dcamf_deepship['si_snri']:.2f}, SDRi={dcamf_deepship['sdri']:.2f}")
else:
    print(f'DeepShip test dir not found: {ds_dir}')
    dcamf_deepship = None


## 6. 基线模型 — 实时加载权重并推理

In [ ]:
# ===== CRN =====
print('Loading CRN...')
sys.path.insert(0, str(PROJECT_ROOT / 'baselines' / 'CRN-causal' / 'scripts'))
from utils.networks import Net

crn_net = Net().to(device)
crn_ckpt_path = PROJECT_ROOT / 'baselines' / 'CRN-causal' / 'scripts' / 'exp' / 'models' / 'best.pt'
crn_ckpt = torch.load(crn_ckpt_path, map_location=device, weights_only=False)
crn_net.load_state_dict(crn_ckpt.net_state_dict)
crn_net.eval()
print(f'CRN loaded. Params: {sum(p.numel() for p in crn_net.parameters()):,}')

# CRN uses STFT-based processing
crn_r = eval_crn_stft(crn_net, str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1'), device, desc='CRN')
print(f"CRN: SI-SNRi={crn_r['si_snri']:.2f}, SDRi={crn_r['sdri']:.2f}")


In [ ]:
# ===== Conv-TasNet =====
print('Loading Conv-TasNet...')
from asteroid.models import ConvTasNet

ct_model = ConvTasNet(n_src=1, sample_rate=16000, n_filters=512,
                       kernel_size=16, stride=8, n_blocks=8, n_repeats=3).to(device)
ct_path = PROJECT_ROOT / 'experiments' / 'conv_tasnet' / 'checkpoints' / 'best_model.pth'
ct_model.load_state_dict(torch.load(ct_path, map_location=device, weights_only=False))
ct_model.eval()
print(f'Conv-TasNet loaded. Params: {sum(p.numel() for p in ct_model.parameters()):,}')

ct_r = eval_model_on_testset(ct_model, str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1'), device, desc='ConvTasNet')
print(f"Conv-TasNet: SI-SNRi={ct_r['si_snri']:.2f}, SDRi={ct_r['sdri']:.2f}")


In [ ]:
# ===== DPRNN =====
print('Loading DPRNN...')
from asteroid.models import DPRNNTasNet

dp_model = DPRNNTasNet(n_src=1, n_filters=64, kernel_size=2, stride=1,
                        bn_chan=128, hid_size=128, rnn_type='LSTM',
                        n_repeats=6, sample_rate=16000).to(device)
dp_path = PROJECT_ROOT / 'experiments' / 'dprnn' / 'checkpoints' / 'best_model.pth'
dp_model.load_state_dict(torch.load(dp_path, map_location=device, weights_only=False))
dp_model.eval()
print(f'DPRNN loaded. Params: {sum(p.numel() for p in dp_model.parameters()):,}')

dp_r = eval_model_on_testset(dp_model, str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1'), device, desc='DPRNN')
print(f"DPRNN: SI-SNRi={dp_r['si_snri']:.2f}, SDRi={dp_r['sdri']:.2f}")


In [ ]:
baseline_results = {
    'CRN': crn_r,
    'Conv-TasNet': ct_r,
    'DPRNN': dp_r,
}
print('\\n=== Baseline Results ===')
for name, r in baseline_results.items():
    print(f"{name:15s}: SI-SNRi={r['si_snri']:7.2f}  SDRi={r['sdri']:7.2f}")


## 7. 四模型对比表

In [ ]:
import pandas as pd

def s(d, k):
    if d is None: return 'N/A'
    return f"{d.get(k, 0):.2f}"

dc1 = dcamf_shipsear.get('test1', {})
results = {
    'Model': ['CRN', 'Conv-TasNet', 'DPRNN', 'DCAMF-Net'],
    'ShipsEar SI-SNRi': [s(baseline_results.get('CRN'),'si_snri'), s(baseline_results.get('Conv-TasNet'),'si_snri'),
                         s(baseline_results.get('DPRNN'),'si_snri'), s(dc1,'si_snri')],
    'ShipsEar SDRi': [s(baseline_results.get('CRN'),'sdri'), s(baseline_results.get('Conv-TasNet'),'sdri'),
                      s(baseline_results.get('DPRNN'),'sdri'), s(dc1,'sdri')],
}
df = pd.DataFrame(results)
def hl(row):
    if row['Model'] == 'DCAMF-Net':
        return ['font-weight: bold; background-color: #D5F0E0'] * len(row)
    return [''] * len(row)
display(df.style.apply(hl, axis=1))
if dcamf_deepship:
    print(f"DeepShip (DCAMF-Net): SI-SNRi={dcamf_deepship['si_snri']:.2f}, SDRi={dcamf_deepship['sdri']:.2f}")


## 8. 试听 + 波形

In [ ]:
clean_d = PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'clean'
noisy_d = PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'noisy'
files = sorted(list(clean_d.glob('*.wav')))
idx = random.randint(0, len(files)-1)
clean, sr = sf.read(str(clean_d / files[idx].name))
noisy, _ = sf.read(str(noisy_d / files[idx].name))
if clean.ndim > 1: clean = clean.mean(axis=1)
if noisy.ndim > 1: noisy = noisy.mean(axis=1)
L = min(len(clean), len(noisy)); clean, noisy = clean[:L], noisy[:L]

noisy_t = torch.from_numpy(noisy).float().unsqueeze(0).unsqueeze(0).to(device)
with torch.no_grad(): s_est = model(noisy_t).squeeze().cpu().numpy()  # model returns enhanced signal
if len(s_est) > L: s_est = s_est[:L]
elif len(s_est) < L: s_est = np.pad(s_est, (0, L-len(s_est)))
enhanced = s_est  # model output IS the enhanced signal

fig, axes = plt.subplots(3, 1, figsize=(14, 7))
t = np.arange(L) / sr
for ax, sig, title, c in zip(axes,
    [clean, noisy, enhanced],
    ['Clean', 'Noisy', 'DCAMF-Net'],
    ['black', 'gray', '#1C7293']):
    ax.plot(t, sig, color=c, linewidth=0.6)
    ax.set_title(title); ax.set_xlim(0, 0.2)
plt.tight_layout(); plt.show()

import IPython.display as ipd
print(f'Sample #{idx}'); ipd.display(ipd.Audio(noisy, rate=sr))
ipd.display(ipd.Audio(enhanced, rate=sr))
ipd.display(ipd.Audio(clean, rate=sr))


## 9. PSD 频谱对比

In [ ]:
from scipy.signal import welch

def psd_db(sig, sr, nperseg=1024):
    f, pxx = welch(sig, sr, nperseg=nperseg, noverlap=nperseg//2)
    return f, 10*np.log10(pxx + 1e-10)

fig, ax = plt.subplots(figsize=(10, 5))
f_c, p_c = psd_db(clean, sr)
f_n, p_n = psd_db(noisy, sr)
f_e, p_e = psd_db(enhanced, sr)
mask = (f_c >= 0) & (f_c <= 4000)
ax.plot(f_c[mask]/1000, p_c[mask], 'k-', lw=1.5, label='Clean')
ax.plot(f_n[mask]/1000, p_n[mask], color='gray', lw=0.6, alpha=0.6, label='Noisy')
ax.plot(f_e[mask]/1000, p_e[mask], '--', color='#1C7293', lw=1.2, label='DCAMF-Net')
ax.set_xlabel('Frequency (kHz)'); ax.set_ylabel('PSD (dB/Hz)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title(f'PSD Comparison (sample #{idx})')
plt.tight_layout(); plt.show()


## 10. 消融实验

In [ ]:
from dcamf_net.model_ablation1 import DCAMFNet_Ablation1
from dcamf_net.model_ablation2 import DCAMFNet_Ablation2
from dcamf_net.model_ablation3 import DCAMFNet_Ablation3

abl_setups = {
    'Full DCAMF-Net': (DCAMFNet, PROJECT_ROOT / 'experiments' / 'dcamf_net' / 'checkpoints' / 'best_SISNR.pth'),
    'No Global Branch': (DCAMFNet_Ablation1, PROJECT_ROOT / 'experiments' / 'ablation' / 'model_ablation1' / 'checkpoints' / 'best_SISNR.pth'),
    'No Local Branch': (DCAMFNet_Ablation2, PROJECT_ROOT / 'experiments' / 'ablation' / 'model_ablation2' / 'checkpoints' / 'best_SISNR.pth'),
    'No Conv Enhance': (DCAMFNet_Ablation3, PROJECT_ROOT / 'experiments' / 'ablation' / 'model_ablation3' / 'checkpoints' / 'best_SISNR.pth'),
}

abl_r = {}
for name, (Cls, ckpt_p) in abl_setups.items():
    if not ckpt_p.exists():
        print(f'{name}: SKIP (checkpoint not found)')
        continue
    m = Cls(in_channels=1, enc_channels=256, enc_kernel_size=80, enc_stride=40,
            chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
            ffn_hidden=512, dw_kernel_size=31).to(device)
    ck = torch.load(ckpt_p, map_location=device, weights_only=False)
    sd = {k: v for k, v in ck.items()
          if not k.endswith('.total_ops') and not k.endswith('.total_params')
          and k not in ('total_ops', 'total_params')}
    m.load_state_dict(sd)
    m.eval()
    r = eval_model_on_testset(m, str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1'), device, desc=f'  {name}')
    abl_r[name] = r
    print(f"{name}: SI-SNRi={r['si_snri']:.2f}, SDRi={r['sdri']:.2f}")

print()
for name, r in abl_r.items():
    print(f'{name:25s}  {r["si_snri"]:7.2f}  {r["sdri"]:7.2f}')


## 11. 融合权重可视化

In [ ]:
log_dir = PROJECT_ROOT / 'experiments' / 'mask_fusion_weights'
groups = {'Average SNR': 'train_avg.log', 'Low SNR': 'train_low.log', 'High SNR': 'train_high.log'}
bar_colors = ['#1C7293', '#999999', '#CCCCCC']

def extract_weights(p):
    if not p.exists(): return None
    for line in reversed(open(p).readlines()):
        if 'MaskFusion Weights (softmax):' in line:
            m = re.search(r'\\[(.+?)\\]', line)
            if m: return [float(x) for x in m.group(1).split(',')]
    return None

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(1, 11)
ok = False
for i, (name, fn) in enumerate(groups.items()):
    w = extract_weights(log_dir / fn)
    if w and len(w) == 10:
        ok = True
        ax.bar(x + i*0.25, w, 0.25, label=name, color=bar_colors[i], edgecolor='black', linewidth=0.5)
        for j, val in enumerate(w):
            if val > 0.01:
                ax.text(x[j]+i*0.25, val, f'{val:.2f}', ha='center', va='bottom', fontsize=7)

if ok:
    ax.set_xlabel('DCAM Block Layer'); ax.set_ylabel('Fusion Weight (softmax)')
    ax.set_title('Multi-Layer Mask Fusion Weights')
    ax.set_xticks(x + 0.25); ax.set_xticklabels([str(i) for i in range(1, 11)])
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print(f'No fusion weight logs found in {log_dir}/')


## 12. 完成清单

In [ ]:
print('Inference Checklist:')
items = [
    'Env check', 'Load DCAMF-Net', 'Test1/2/3 evaluation',
    'DeepShip evaluation',
    'CRN loaded & evaluated (real-time)',
    'Conv-TasNet loaded & evaluated (real-time)',
    'DPRNN loaded & evaluated (real-time)',
    'Comparison table', 'Audio preview',
    'PSD comparison', 'Ablation results', 'Fusion weights plot'
]
for i, item in enumerate(items):
    print(f'  [{i+1}] {item}')
print('\\nAll models loaded and evaluated in real-time — no pre-computed files used!')
